In [3]:
import pandas as pd
import numpy as np

### Dagshub/Mlflow initialization ###

In [4]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner='sbolk23', repo_name='House-Prices-Kaggle-Competition', mlflow=True)


Accessing as sbolk23

Initialized MLflow to track repo "sbolk23/House-Prices-Kaggle-Competition"

Repository sbolk23/House-Prices-Kaggle-Competition initialized!

In [5]:
PATH = './data'
TARGET = 'SalePrice'

In [6]:
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

### Split Train/Test Data ###

In [7]:
df = pd.read_csv(PATH + '/train.csv')
# df = df.drop([523, 1298], axis=0)

In [8]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, shuffle=True, random_state=1337
)

train_ids = X_train['Id']
test_ids = X_test['Id']

print(f'X_train shape {X_train.shape}')
print(f'X_test shape {X_test.shape}')

X_train shape (1168, 80)
X_test shape (292, 80)


### Data Cleaning ###

### Categorical Imputing ###

In [9]:
# we first impute all missing values in categorical and numeric features.

# These categorical columns may have 'NaN' values in them which means missing.
# data_description.txt did not mention 'Nan' was intended.

cat_impute_cols = [
    'MSZoning',
    'Street',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'Foundation',
    'ExterQual',
    'ExterCond',
    'Heating',
    'HeatingQC',
    'CentralAir',
    'Electrical',
    'KitchenQual',
    'Functional',
    'PavedDrive',
    'SaleType',
    'SaleCondition',
]

### Numeric Imputing ###

In [10]:
# We impute all numeric columns because data_description.txt did not mention any numeric feature with intended 'NaN' value.

# num_impute_cols = [
#     'LotFrontage',
#     'GarageYrBlt',
#     'MasVnrArea',
# ]

num_impute_cols = list(X_train.select_dtypes(exclude='object').columns)
num_impute_cols.remove('Id')
num_impute_cols.remove('MSSubClass')

### Categorical One-Hot-Encoded Columns ###

In [11]:
# Here we list features to be one-hot-encoded including 'NaN's because 'NaN' here does not mean missing.
# 'MasVnrType' was interesting column here.

cat_ohe_cols = [
    'MSZoning',
    'Street',
    'Alley',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'MasVnrType',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'GarageType',
    'PavedDrive',
    'MiscFeature',
    'SaleType',
    'SaleCondition',
]

cat_ohe_missing_cols = [
    'MSZoning',
    'Street',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'PavedDrive',
    'SaleType',
    'SaleCondition'
]

cat_ohe_absent_cols = [
    'Alley',
    'MasVnrType',
    'GarageType',
    'MiscFeature'
]

### Ordinal Columns ###

In [12]:
# Some categorical columns can may be ordered, replace them with numbers.

cat_ord_cols = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'HeatingQC',
    'KitchenQual',
    'Functional',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

cat_ord_missing_cols = [
    'ExterQual',
    'ExterCond',
    'HeatingQC',
    'KitchenQual',
    'Functional',
]

cat_ord_absent_cols = [
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

exter_qual_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
exter_cond_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_qual_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_cond_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_exposure_order     = ['NA', 'No', 'Mn', 'Av', 'Gd']
bsmt_fin_type_1_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
bsmt_fin_type_2_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
heating_qc_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
kitchen_qual_order      = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
functional_order        = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
fireplace_qu_order      = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_finish_order     = ['NA', 'Unf', 'RFn', 'Fin']
garage_qual_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_cond_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
pool_qc_order           = ['NA', 'Fa', 'TA', 'Gd', 'Ex']
fence_order             = ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']

cat_ord_missing_categories = [
    exter_qual_order,
    exter_cond_order,
    heating_qc_order,
    kitchen_qual_order,
    functional_order,
]

cat_ord_absent_categories = [
    bsmt_qual_order,
    bsmt_cond_order,
    bsmt_exposure_order,
    bsmt_fin_type_1_order,
    bsmt_fin_type_2_order,
    fireplace_qu_order,
    garage_finish_order,
    garage_qual_order,
    garage_cond_order,
    pool_qc_order,
    fence_order,
]

### Numeric One-Hot-Encoded Columns ###

In [13]:
# This feature has numeric type which is misleading.

num_ohe_cols = [
    'MSSubClass',
]

### Irrelevant Columns ###

In [14]:
# This col is not informative.

irrelevant_cols = [
    'Id'
]

### Data Cleanup Pipeline ###

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# scaling every column after preprocessing decreased accuracy.
# scaling only numeric columns is sufficient.

num_impute_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='median')),
    ('scale',   StandardScaler())
])

num_ohe_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cat_ord_missing_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=cat_ord_missing_categories)),
])

cat_ord_absent_pipeline = Pipeline([
    ('nan_impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='NA')),
    ('ordinal',     OrdinalEncoder(categories=cat_ord_absent_categories)),
])

cat_ohe_missing_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

cat_ohe_absent_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='Missing')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num_impute',       num_impute_pipeline,       num_impute_cols),
        ('num_ohe',          num_ohe_pipeline,          num_ohe_cols),
        ('cat_ord_absent',   cat_ord_absent_pipeline,   cat_ord_absent_cols),
        ('cat_ord_missing',  cat_ord_missing_pipeline,  cat_ord_missing_cols),
        ('cat_ohe_absent',   cat_ohe_absent_pipeline,   cat_ohe_absent_cols),
        ('cat_ohe_missing',  cat_ohe_missing_pipeline,  cat_ohe_missing_cols),
        ('drop',            'drop',                     irrelevant_cols),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

preprocessor.set_output(transform='pandas')

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_impute', ...), ('num_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

### Custom Scorer ###

In [16]:
from sklearn.metrics import make_scorer, root_mean_squared_log_error

def safe_rmsle(y_true, y_pred):
    y_pred_pos = np.maximum(y_pred, 0)
    return root_mean_squared_log_error(y_true, y_pred_pos)

safe_rmsle_scorer = make_scorer(safe_rmsle, greater_is_better=False)

### Full Pipeline (Linear Regression) ###

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

final_model = TransformedTargetRegressor(
    regressor=pipeline,
)

param_grid = [
    {
        'regressor__preprocessor__num_impute__impute__strategy': 'mean',
        'func': np.log1p,
        'inverse_func': np.expm1
    },
    {
        'regressor__preprocessor__num_impute__impute__strategy': 'mean',
        'func': None,
        'inverse_func': None
    },
    {
        'regressor__preprocessor__num_impute__impute__strategy': 'median',
        'func': np.log1p,
        'inverse_func': np.expm1
    },
    {
        'regressor__preprocessor__num_impute__impute__strategy': 'median',
        'func': None,
        'inverse_func': None
    },
]

### MLFlow Logging (Linear Regression) ###

In [16]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.base import clone

kf = KFold(n_splits=5, shuffle=True, random_state=1337)


mlflow.set_experiment('linear_regression')

for params in param_grid:
    num_imp = params['regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if params['func'] is not None else "rawY"

    run_name = f'LR__prep_v1__{has_log}__num_imp_{num_imp}__ord_imp__ohe'

    with mlflow.start_run(run_name=run_name):
        model = clone(final_model)
        model.set_params(**params)

        cv_results = cross_validate(
            model, X_train, y_train,
            cv=kf,
            scoring={
                'neg_root_mean_squared_log_error': safe_rmsle_scorer,
                'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
                'neg_mean_absolute_error': 'neg_mean_absolute_error',
                'r2': 'r2',
            },
            return_train_score=True
        )

        print(f'Logging parameters ({run_name})')
        for key, value in params.items():
            mlflow.log_param(key, value)

        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))


        print(f'Logging train and validation metrics ({run_name})')

        mlflow.log_metric('mean_train_rmsle',        abs(np.array(cv_results['train_neg_root_mean_squared_log_error']).mean()))
        mlflow.log_metric('mean_validation_rmsle',   abs(np.array(cv_results['test_neg_root_mean_squared_log_error']).mean()))
        mlflow.log_metric('std_train_rmsle',         (np.array(cv_results['train_neg_root_mean_squared_log_error']).std()))
        mlflow.log_metric('std_validation_rmsle',    (np.array(cv_results['test_neg_root_mean_squared_log_error']).std()))

        mlflow.log_metric('mean_train_rmse',        abs(np.array(cv_results['train_neg_root_mean_squared_error']).mean()))
        mlflow.log_metric('mean_validation_rmse',   abs(np.array(cv_results['test_neg_root_mean_squared_error']).mean()))
        mlflow.log_metric('std_train_rmse',         (np.array(cv_results['train_neg_root_mean_squared_error']).std()))
        mlflow.log_metric('std_validation_rmse',    (np.array(cv_results['test_neg_root_mean_squared_error']).std()))

        mlflow.log_metric('mean_train_mae',         abs(np.array(cv_results['train_neg_mean_absolute_error']).mean()))
        mlflow.log_metric('mean_validation_mae',    abs(np.array(cv_results['test_neg_mean_absolute_error']).mean()))
        mlflow.log_metric('std_train_mae',          (np.array(cv_results['train_neg_mean_absolute_error']).std()))
        mlflow.log_metric('std_validation_mae',     (np.array(cv_results['test_neg_mean_absolute_error']).std()))

        mlflow.log_metric('mean_train_r2',          (np.array(cv_results['train_r2']).mean()))
        mlflow.log_metric('mean_validation_r2',     (np.array(cv_results['test_r2']).mean()))
        mlflow.log_metric('std_train_r2',           (np.array(cv_results['train_r2']).std()))
        mlflow.log_metric('std_validation_r2',      (np.array(cv_results['test_r2']).std()))


        for i in range(5):
            mlflow.log_metric(f'split{i}_train_rmsle',       abs(cv_results['train_neg_root_mean_squared_log_error'][i]))
            mlflow.log_metric(f'split{i}_validation_rmsle',  abs(cv_results['test_neg_root_mean_squared_log_error'][i]))
            mlflow.log_metric(f'split{i}_train_rmse',        abs(cv_results['train_neg_root_mean_squared_error'][i]))
            mlflow.log_metric(f'split{i}_validation_rmse',   abs(cv_results['test_neg_root_mean_squared_error'][i]))
            mlflow.log_metric(f'split{i}_train_mae',         abs(cv_results['train_neg_mean_absolute_error'][i]))
            mlflow.log_metric(f'split{i}_validation_mae',    abs(cv_results['test_neg_mean_absolute_error'][i]))
            mlflow.log_metric(f'split{i}_train_r2',          (cv_results['train_r2'][i]))
            mlflow.log_metric(f'split{i}_validation_r2',     (cv_results['test_r2'][i]))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      np.array(cv_results['fit_time']).mean())
        mlflow.log_metric('std_fit_time',       np.array(cv_results['fit_time']).std())
        mlflow.log_metric('mean_score_time',    np.array(cv_results['score_time']).mean())
        mlflow.log_metric('std_score_time',     np.array(cv_results['score_time']).std())

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('model_id', model_info.model_id)


Logging parameters (LR__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (LR__prep_v1__logY__num_imp_mean__ord_imp__ohe)
🏃 View run LR__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/5148738ded314a36b3e6831698c18aad
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1


KeyboardInterrupt: 

### Full Pipeline (Ridge) ###

In [24]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

final_model = TransformedTargetRegressor(
    regressor=pipeline,
)

param_grid = [
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
        'regressor__model__alpha': [.001, .01, .1,  1, 5, 10, 25, 50, 100, 1000],
        'func': [np.log1p],
        'inverse_func': [np.expm1],
    },
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
        'regressor__model__alpha': [.001, .01, .1,  1, 5, 10, 25, 50, 100, 1000],
        'func': [None],
        'inverse_func': [None],
    },

    # 'regressor__model__alpha': np.linspace(.01, 1000, 300),
    # 'regressor__model__alpha': np.logspace(np.log10(9.5), np.log10(10), 300),
]

In [25]:
from sklearn.model_selection import KFold
from sklearn.base import clone

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

grid_search_cv = GridSearchCV(
    final_model,
    param_grid,
    cv=kf,
    scoring={
        'neg_root_mean_squared_log_error': safe_rmsle_scorer,
        'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
        'neg_mean_absolute_error': 'neg_mean_absolute_error',
        'r2': 'r2',
    },
    refit='neg_root_mean_squared_log_error',
    return_train_score=True,
    n_jobs=-1
)

grid_search_cv.fit(X_train, y_train)
cv_results_df = pd.DataFrame(grid_search_cv.cv_results_)

mlflow.set_experiment('linear_regression')

for i, row in cv_results_df.iterrows():
    row_dict = row.to_dict()

    print(row_dict)

    params = row_dict['params']
    alpha = row_dict['param_regressor__model__alpha']
    num_imp = row_dict['param_regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if row_dict['param_func'] is not None else "rawY"

    run_name = f'RIDGE__alpha_{alpha}__prep_v1__{has_log}__num_imp_{num_imp}__ord_imp__ohe'

    with mlflow.start_run(run_name=run_name):
        print(f'Logging parameters ({run_name})')
        for key, value in row_dict.items():
            if 'param_' in key:
                mlflow.log_param(key.replace('param_', ''), value)

        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))

        print(f'Logging train and validation metrics ({run_name})')
        for key, value in row_dict.items():
            if 'rank' in key:
                continue

            if 'neg_root_mean_squared_log_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_log_error', 'rmsle').replace('test', 'validation'), abs(value))

            if 'neg_root_mean_squared_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_error', 'rmse').replace('test', 'validation'), abs(value))

            if 'neg_mean_absolute_error' in key:
                mlflow.log_metric(
                    key.replace('neg_mean_absolute_error', 'mae').replace('test', 'validation'), abs(value))

            if 'r2' in key:
                mlflow.log_metric(key.replace('test', 'validation'), (value))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      row_dict['mean_fit_time'])
        mlflow.log_metric('std_fit_time',       row_dict['std_fit_time'])
        mlflow.log_metric('mean_score_time',    row_dict['mean_score_time'])
        mlflow.log_metric('std_score_time',     row_dict['std_score_time'])

        model = clone(final_model)
        model.set_params(**params)

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('model_id', model_info.model_id)


{'mean_fit_time': 0.13090081214904786, 'std_fit_time': 0.0037938894402134633, 'mean_score_time': 0.07557978630065917, 'std_score_time': 0.014778919163126203, 'param_func': <ufunc 'log1p'>, 'param_inverse_func': <ufunc 'expm1'>, 'param_regressor__model__alpha': 0.001, 'param_regressor__preprocessor__num_impute__impute__strategy': 'mean', 'params': {'func': <ufunc 'log1p'>, 'inverse_func': <ufunc 'expm1'>, 'regressor__model__alpha': 0.001, 'regressor__preprocessor__num_impute__impute__strategy': 'mean'}, 'split0_test_neg_root_mean_squared_log_error': -0.1632601294549023, 'split1_test_neg_root_mean_squared_log_error': -0.14243554151911755, 'split2_test_neg_root_mean_squared_log_error': -0.20397260326353656, 'split3_test_neg_root_mean_squared_log_error': -0.17187506026362612, 'split4_test_neg_root_mean_squared_log_error': -0.18222570103291943, 'mean_test_neg_root_mean_squared_log_error': -0.1727538071068204, 'std_test_neg_root_mean_squared_log_error': 0.020368747372586794, 'rank_test_neg_r

2026/04/10 18:52:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RIDGE__alpha_0.001__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/62d59250dfd44b1a87f0c626ccb37e23
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
{'mean_fit_time': 0.1491565227508545, 'std_fit_time': 0.02017520726537496, 'mean_score_time': 0.0652618408203125, 'std_score_time': 0.0031590298958125112, 'param_func': <ufunc 'log1p'>, 'param_inverse_func': <ufunc 'expm1'>, 'param_regressor__model__alpha': 0.001, 'param_regressor__preprocessor__num_impute__impute__strategy': 'median', 'params': {'func': <ufunc 'log1p'>, 'inverse_func': <ufunc 'expm1'>, 'regressor__model__alpha': 0.001, 'regressor__preprocessor__num_impute__impute__strategy': 'median'}, 'split0_test_neg_root_mean_squared_log_error': -0.16322408877151753, 'split1_test_neg_root_mean_squared_log_error': -0.14240678300978496, 'split2_test_neg_root_mean_squared_log_error': -0.

2026/04/10 18:54:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RIDGE__alpha_0.001__prep_v1__logY__num_imp_median__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/05275345d087416aabab5b9ec48beb4a
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
{'mean_fit_time': 0.18865504264831542, 'std_fit_time': 0.059688689266513585, 'mean_score_time': 0.0638033390045166, 'std_score_time': 0.004218630098340344, 'param_func': <ufunc 'log1p'>, 'param_inverse_func': <ufunc 'expm1'>, 'param_regressor__model__alpha': 0.01, 'param_regressor__preprocessor__num_impute__impute__strategy': 'mean', 'params': {'func': <ufunc 'log1p'>, 'inverse_func': <ufunc 'expm1'>, 'regressor__model__alpha': 0.01, 'regressor__preprocessor__num_impute__impute__strategy': 'mean'}, 'split0_test_neg_root_mean_squared_log_error': -0.16189534183126453, 'split1_test_neg_root_mean_squared_log_error': -0.14201933793507412, 'split2_test_neg_root_mean_squared_log_error': -0.203

...

🏃 View run RIDGE__alpha_100.0__prep_v1__rawY__num_imp_median__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/85808fce989f447ca5aa1cae10056993
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
{'mean_fit_time': 0.3453166961669922, 'std_fit_time': 0.06666541227651375, 'mean_score_time': 0.14213061332702637, 'std_score_time': 0.010784083944068776, 'param_func': None, 'param_inverse_func': None, 'param_regressor__model__alpha': 1000.0, 'param_regressor__preprocessor__num_impute__impute__strategy': 'mean', 'params': {'func': None, 'inverse_func': None, 'regressor__model__alpha': 1000, 'regressor__preprocessor__num_impute__impute__strategy': 'mean'}, 'split0_test_neg_root_mean_squared_log_error': -0.16145967881978746, 'split1_test_neg_root_mean_squared_log_error': -0.1650825767281046, 'split2_test_neg_root_mean_squared_log_error': -0.17983837721431295, 'split3_test_neg_root_mean_s

2026/04/10 20:25:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RIDGE__alpha_1000.0__prep_v1__rawY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/049e8a2d4dd248e881aa55b6e857f0ad
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
{'mean_fit_time': 0.3234294891357422, 'std_fit_time': 0.013042760239952173, 'mean_score_time': 0.13174376487731934, 'std_score_time': 0.019831393015287548, 'param_func': None, 'param_inverse_func': None, 'param_regressor__model__alpha': 1000.0, 'param_regressor__preprocessor__num_impute__impute__strategy': 'median', 'params': {'func': None, 'inverse_func': None, 'regressor__model__alpha': 1000, 'regressor__preprocessor__num_impute__impute__strategy': 'median'}, 'split0_test_neg_root_mean_squared_log_error': -0.161501542536962, 'split1_test_neg_root_mean_squared_log_error': -0.1649819337984061, 'split2_test_neg_root_mean_squared_log_error': -0.1799165133553077, 'split3_test_neg_root_mean_

2026/04/10 20:28:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RIDGE__alpha_1000.0__prep_v1__rawY__num_imp_median__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/bf9695052586490a8dc05359c9a3f147
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1


### Full Pipeline (Lasso) ###

In [123]:
# from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Lasso(max_iter=10000))
])

param_grid = {
    'preprocessor__num_impute__impute__strategy': ['mean', 'median'],
    # 'model__alpha': [.0005, .0006, .0007, .0008, .0009, .001, .005, .01, .05, .1, .5, 1, 5, 10],
    'model__alpha': np.logspace(-4, 1, 300),
}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring={
        'neg_root_mean_squared_log_error': safe_rmsle_scorer,
        'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
        'neg_mean_absolute_error': 'neg_mean_absolute_error',
        'r2': 'r2',
    },
    refit='neg_root_mean_squared_log_error',
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

search.fit(X_train, y_train)
cv_results_df = pd.DataFrame(search.cv_results_)

cv_results_df

Fitting 5 folds for each of 600 candidates, totalling 3000 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,param_preprocessor__preprocessor__num_impute__impute__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.798562,0.288849,0.192542,0.053236,0.000100,mean,"{'model__alpha': 0.0001, 'preprocessor__prepro...",-0.156672,-0.126751,-0.205040,-0.131516,-0.151069,-0.154210,0.027816,184,-0.091772,-0.092366,-0.093926,-0.093994,-0.088841,-0.092180,0.001881
1,0.494709,0.080717,0.153101,0.040209,0.000100,median,"{'model__alpha': 0.0001, 'preprocessor__prepro...",-0.156628,-0.126733,-0.205042,-0.131520,-0.151075,-0.154200,0.027819,183,-0.091784,-0.092372,-0.093927,-0.093998,-0.088841,-0.092184,0.001882
2,0.605070,0.101625,0.159681,0.027988,0.000104,mean,"{'model__alpha': 0.00010392556829367131, 'prep...",-0.156585,-0.126356,-0.205113,-0.130929,-0.150560,-0.153909,0.028027,178,-0.091921,-0.092515,-0.094042,-0.094112,-0.089000,-0.092318,0.001865
3,0.627594,0.091575,0.162277,0.035550,0.000104,median,"{'model__alpha': 0.00010392556829367131, 'prep...",-0.156541,-0.126338,-0.205119,-0.130930,-0.150566,-0.153899,0.028031,177,-0.091933,-0.092521,-0.094042,-0.094116,-0.089000,-0.092322,0.001866
4,0.411241,0.058485,0.140560,0.040918,0.000108,mean,"{'model__alpha': 0.00010800523745162529, 'prep...",-0.156482,-0.125957,-0.205170,-0.130335,-0.150103,-0.153609,0.028232,172,-0.092078,-0.092674,-0.094143,-0.094230,-0.089164,-0.092458,0.001845
5,0.460329,0.098281,0.147776,0.047190,0.000108,median,"{'model__alpha': 0.00010800523745162529, 'prep...",-0.156440,-0.125939,-0.205177,-0.130338,-0.150107,-0.153600,0.028237,171,-0.092090,-0.092680,-0.094143,-0.094234,-0.089163,-0.092462,0.001846
6,0.462331,0.079025,0.160286,0.045095,0.000112,mean,"{'model__alpha': 0.0001122450568085307, 'prepr...",-0.156337,-0.125553,-0.205220,-0.129715,-0.149639,-0.153293,0.028441,168,-0.092240,-0.092846,-0.094238,-0.094355,-0.089328,-0.092601,0.001825
7,0.455558,0.073185,0.119823,0.008281,0.000112,median,"{'model__alpha': 0.0001122450568085307, 'prepr...",-0.156304,-0.125536,-0.205228,-0.129724,-0.149644,-0.153287,0.028445,167,-0.092252,-0.092852,-0.094238,-0.094359,-0.089328,-0.092606,0.001826
8,0.471180,0.173373,0.122983,0.003217,0.000117,mean,"{'model__alpha': 0.00011665131316981962, 'prep...",-0.156127,-0.125137,-0.205257,-0.129080,-0.149141,-0.152949,0.028650,164,-0.092409,-0.093030,-0.094335,-0.094490,-0.089469,-0.092747,0.001817
9,0.557206,0.164909,0.121148,0.003691,0.000117,median,"{'model__alpha': 0.00011665131316981962, 'prep...",-0.156096,-0.125121,-0.205262,-0.129090,-0.149147,-0.152943,0.028652,163,-0.092422,-0.093036,-0.094336,-0.094495,-0.089469,-0.092751,0.001817


### Final Pipeline (ElasticNet) ###

In [81]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', ElasticNet(max_iter=10000))
])

param_grid = {
    'preprocessor__num_impute__impute__strategy':
        ['mean', 'median'],
    # 'model__alpha':
    #     [.0005, .001, .002, .005, .1, .5, 1],
    # 'model__l1_ratio':
    #     [.1, .2, .3, .4, .5, .6, .7, .8, .9],

    'model__alpha':
        np.logspace(-4, 1, 30),
    'model__l1_ratio':
        np.linspace(.05, .95, 20)
}

# {\'model__alpha\': 0.0007278953843983154, \'model__l1_ratio\': 0.7131578947368421, \'preprocessor__preprocessor__num_impute__impute__strategy\': \'mean\'}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
    verbose=1,
    n_jobs=-1,
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)
cv_results_df

Fitting 5 folds for each of 1200 candidates, totalling 6000 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,param_model__l1_ratio,param_preprocessor__preprocessor__num_impute__impute__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,3.314577,3.392029,0.261353,0.102276,0.000100,0.050000,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.142375,-0.137402,-0.130484,-0.106842,-0.124139,-0.128248,0.012357,569,-0.081123,-0.083706,-0.085303,-0.089008,-0.085955,-0.085019,0.002599
1,3.442399,2.941844,0.192308,0.057177,0.000100,0.050000,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.142391,-0.137390,-0.130474,-0.106837,-0.124179,-0.128254,0.012358,570,-0.081118,-0.083703,-0.085311,-0.089008,-0.085951,-0.085018,0.002601
2,2.970581,2.475001,0.202630,0.048384,0.000100,0.097368,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.142077,-0.136683,-0.129734,-0.106614,-0.123047,-0.127631,0.012311,557,-0.081194,-0.083776,-0.085358,-0.089078,-0.086018,-0.085085,0.002598
3,2.901912,2.743442,0.165322,0.021030,0.000100,0.097368,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.142090,-0.136672,-0.129723,-0.106611,-0.123089,-0.127637,0.012310,558,-0.081189,-0.083773,-0.085365,-0.089079,-0.086013,-0.085084,0.002600
4,2.308816,2.204081,0.221125,0.096675,0.000100,0.144737,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.141703,-0.135960,-0.129092,-0.106259,-0.122393,-0.127081,0.012265,547,-0.081276,-0.083859,-0.085419,-0.089161,-0.086085,-0.085160,0.002596
5,2.324319,1.797121,0.164558,0.032120,0.000100,0.144737,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.141714,-0.135950,-0.129075,-0.106256,-0.122427,-0.127085,0.012265,548,-0.081271,-0.083857,-0.085426,-0.089162,-0.086081,-0.085159,0.002598
6,0.782089,0.147618,0.160000,0.036521,0.000100,0.192105,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.141331,-0.135335,-0.128693,-0.105867,-0.121816,-0.126608,0.012251,541,-0.081369,-0.083954,-0.085487,-0.089250,-0.086162,-0.085245,0.002593
7,0.700925,0.133491,0.168572,0.035241,0.000100,0.192105,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.141340,-0.135326,-0.128675,-0.105863,-0.121848,-0.126610,0.012250,542,-0.081364,-0.083952,-0.085495,-0.089251,-0.086159,-0.085244,0.002595
8,0.659668,0.188697,0.134308,0.012748,0.000100,0.239474,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.140978,-0.134776,-0.128283,-0.105502,-0.121283,-0.126164,0.012238,525,-0.081472,-0.084059,-0.085558,-0.089341,-0.086237,-0.085334,0.002587
9,0.550341,0.162798,0.135884,0.024841,0.000100,0.239474,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.140987,-0.134767,-0.128265,-0.105499,-0.121315,-0.126167,0.012237,526,-0.081467,-0.084056,-0.085566,-0.089342,-0.086234,-0.085333,0.002589


In [121]:
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', ElasticNet(alpha=0.0007278953843983154, l1_ratio=0.7131578947368421))
])
final_pipeline.fit(X_train, y_train_t)
y_pred = final_pipeline.predict(X_test)
print(root_mean_squared_error(y_test_t, y_pred))

y_pred = final_pipeline.predict(X_train)
print(root_mean_squared_error(y_train_t, y_pred))



# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', Lasso(alpha=.0007934096665797492))
# ])
# final_pipeline.fit(X_train, y_train_t)
# y_pred = final_pipeline.predict(X_test)
# print(root_mean_squared_error(y_test_t, y_pred))
#
# y_pred = final_pipeline.predict(X_train)
# print(root_mean_squared_error(y_train_t, y_pred))
#
#
# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', LinearRegression())
# ])
# final_pipeline.fit(X_train, y_train_t)
# y_pred = final_pipeline.predict(X_test)
# print(root_mean_squared_error(y_test_t, y_pred))
#
# y_pred = final_pipeline.predict(X_train)
# print(root_mean_squared_error(y_train_t, y_pred))

0.11544915376776345
0.10881045553882318
0.1156369745205326
0.11243910374712052
0.12966410509994647
0.09419088615888568


In [189]:
final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Ridge(alpha=400))
])

cv_results = cross_validate(
    final_pipeline, X_train, y_train_t,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
)

cv_results_df = pd.DataFrame(cv_results)

train_root_mean_squared_error = -cv_results_df.mean()['train_score']
validation_root_mean_squared_error = -cv_results_df.mean()['test_score']

print(f'train_rmse: {train_root_mean_squared_error}')
print(f'validation_rmse: {validation_root_mean_squared_error}')

train_rmse: 0.10310912196089887
validation_rmse: 0.13957445547139208


In [124]:
# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', ElasticNet(alpha=0.0007278953843983154, l1_ratio=0.7131578947368421))
# ])

# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', Ridge(alpha=9.846791241561434))
# ])

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Lasso(alpha=0.0006348565216305217))
])

test = pd.read_csv(PATH + '/test.csv')
test_ids = test['Id']

# final_pipeline.fit(pd.concat([X_train, X_test], axis=0, ignore_index=True), pd.concat([y_train_t, y_test_t], axis=0, ignore_index=True))
final_pipeline.fit(X_train, y_train_t)
prices_log = final_pipeline.predict(test)
prices = np.expm1(prices_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': prices,
})

submission.to_csv('submission.csv', index=False)

### Final Test scores ###

In [61]:
from sklearn.metrics import root_mean_squared_error

runs_df = mlflow.search_runs(experiment_names=["linear_regression"])

for run_id in runs_df["run_id"]:
    models = mlflow.search_logged_models(
        filter_string=f"source_run_id = '{run_id}'",
        output_format="list",
    )

    model_id = models[0].model_id
    model = mlflow.sklearn.load_model(f"models:/{model_id}")
    y_pred = model.predict(X_test)

    test_rmse = root_mean_squared_error(y_test_t, y_pred)

    with mlflow.start_run(run_id=run_id):
        mlflow.log_metric('test_rmse', test_rmse)


🏃 View run LR__prep_v1__num_imp_median__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/f7155e7f2be3435e919d5aab2e1b0bf5
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1


🏃 View run LR__prep_v1__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/339189ffc2fd4096a73dc763dce0fffb
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
